# Deep **justos** (FM / DeepFM / DeepNN) sobre **KOZYRIEV** — protocolo H3 (5-core, full-ranking, sin fuga)

**Versión 2 de las dos versiones deep para Kozyriev** (la otra es `cheuque_kozyriev.ipynb` = filtro denso + protocolo leaky de Cheuque).

Corre los 3 modelos *feature-based* de Cheuque **bajo NUESTRO protocolo H3**, igual que ya están dentro de `CPGRec_comparativo_multiseed_ucsd_corregido.ipynb` para UCSD:
- **Mismo prep determinista** que `CPGRec_comparativo_multiseed_kozyriev_corregido` (k-core 5/20, dedup de pares, subsample 500k, split random 80/10/10, `eval_set` fijo de 200k, `paper_metrics`) → **las filas FM/DeepFM/DeepNN caen directo en las tablas §A/§B/§C/§D** junto a MostPop/ALS/CPGRec.
- **Positivos implícitos + neg-sampling** (no el label `playtime≥5h`), **SIN playtime como feature** (sin fuga), eval **full-ranking** sobre los ~22K juegos. **1 seed {42}** — multi-seed desactivado: el full-ranking ×3 modelos es muy pesado (las tablas §A/§B/§C salen como valor único, con ±0.0000).

Análisis incluido (portado del comparativo UCSD): **§A accuracy · §B diversidad (Cov/Ent categoría) · §C long-tail por actividad · §D ejemplos con nombres · respaldo que IMPRIME TODO** a stdout (por si no se pueden descargar archivos).

> **DGL-free a propósito:** sin torch 2.4 + DGL ni reinicio de runtime. El split usa `np.random.default_rng(SEED)` → `eval_set` idéntico al del corregido y MostPop/ALS deben reproducir sus filas (self-check).

> **Costo:** full-ranking sobre ~22K juegos × 200k usuarios × 3 modelos (1 seed) es **pesado** en GPU → correr **T1 primero** (`TIER='T1'`: eval cap 2000, 1 época, 1 seed) y el final en A100/L4 (Pro/Pro+). En T4 gratis: solo T1.

In [ ]:
# Setup (DGL-free). Deep = deepctr-torch (usa el torch de Colab; --no-deps no toca versiones).
!pip install -q kagglehub implicit
!pip install -q deepctr-torch --no-deps
import os, sys, glob, json, math, time, csv, zipfile, shutil, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import kagglehub
warnings.filterwarnings('ignore')

SEED = 42
import random
random.seed(SEED); np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except Exception:
    pass
print('setup OK | numpy', np.__version__, '| pandas', pd.__version__)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 104.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.0 MB/s eta 0:00:00
setup OK | numpy 2.0.2 | pandas 2.2.2


In [ ]:
# ===== Config (espeja el prep del corregido de Kozyriev; sin params del GNN) =====
SLUG         = 'antonkozyriev/game-recommendations-on-steam'
FRONKON_SLUG = 'fronkongames/steam-games-dataset'
FILTER_MODE  = 'clasico'                       # 'clasico'=(5,20) | 'agresivo'=(20,50)
_PRESETS     = {'clasico': (5, 20), 'agresivo': (20, 50)}
MIN_USER_REVIEWS, MIN_GAME_REVIEWS = _PRESETS[FILTER_MODE]

KS = (5, 10)
ALS_FACTORS, ALS_ITERS, ALS_REG, ALS_ALPHA = 64, 15, 0.1, 40.0   # mismo ALS que el corregido
SEEDS = [42]                                    # 1 sola seed para FM/DeepFM/DeepNN: full-ranking x3 modelos es muy pesado (poner [42, 1, 2] para multi-seed)
N_EVAL_ANALYSIS = 200_000                       # mismo eval_set que el corregido
N_EXAMPLES = 3                                   # usuarios de ejemplo en §D

TIER = 'T2'                                      # 'T1' = sanity (eval cap 2000, 1 ep, 1 seed)
SUBSAMPLE_USERS = 500_000 if TIER == 'T2' else 200_000
print(f'V2 deep justo | Kozyriev | TIER={TIER} | filtro={FILTER_MODE}{_PRESETS[FILTER_MODE]} | '
      f'subsample={SUBSAMPLE_USERS} | SEEDS={SEEDS} | N_EVAL={N_EVAL_ANALYSIS}')

V2 deep justo | Kozyriev | TIER=T2 | filtro=clasico(5, 20) | subsample=500000 | SEEDS=[42] | N_EVAL=200000


In [ ]:
def _find_file(root, *names):
    cand = {n.lower() for n in names}
    for p in Path(root).rglob('*'):
        if p.is_file() and p.name.lower() in cand:
            return p
    raise FileNotFoundError(f'No encontré {names} bajo {root}')

def _maybe_unzip(p, *wanted):
    with open(p, 'rb') as f:
        if f.read(4) != b'PK\x03\x04':
            return p
    ext = p.parent / (p.stem + '_extracted'); ext.mkdir(exist_ok=True)
    with zipfile.ZipFile(p) as z:
        names = z.namelist(); target = None
        for w in wanted:
            target = next((n for n in names if Path(n).name.lower() == w.lower()), None)
            if target: break
        if target is None:
            target = next((n for n in names if n.lower().endswith(('.csv', '.json'))), names[0])
        out = ext / Path(target).name
        if not out.exists() or out.stat().st_size == 0:
            with z.open(target) as s, open(out, 'wb') as d: shutil.copyfileobj(s, d)
        return out

def fetch_file(slug, *filenames):
    for fn in filenames:
        try:
            p = Path(kagglehub.dataset_download(slug, path=fn))
            p = p if p.is_file() else _find_file(p, fn)
            return _maybe_unzip(p, *filenames)
        except Exception:
            continue
    return _find_file(Path(kagglehub.dataset_download(slug)), *filenames)

def read_csv_robust(path, **kw):
    for enc in ('utf-8', 'utf-8-sig', 'cp1252', 'latin-1'):
        try:
            return pd.read_csv(path, encoding=enc, **kw)
        except (UnicodeDecodeError, UnicodeError):
            continue
    return pd.read_csv(path, encoding='latin-1', encoding_errors='replace', **kw)

t0 = time.time()
games_csv = fetch_file(SLUG, 'games.csv')
meta_json = fetch_file(SLUG, 'games_metadata.json')
recs_csv  = fetch_file(SLUG, 'recommendations.csv')
fg_csv    = fetch_file(FRONKON_SLUG, 'games.csv')
print(f'Descargado en {time.time()-t0:.0f}s')

Using Colab cache for faster access to the 'game-recommendations-on-steam' dataset.
Using Colab cache for faster access to the 'game-recommendations-on-steam' dataset.
Using Colab cache for faster access to the 'game-recommendations-on-steam' dataset.
Using Colab cache for faster access to the 'steam-games-dataset' dataset.
Descargado en 12s


In [ ]:
# ====== Carga recommendations.csv + k-core + positivos + re-split RANDOM 80/10/10 ======
# Interaccion = (user, juego) con is_recommended=True (positivo implicito, playtime=horas),
# paralelo al protocolo implicito de SCGRec. k-core iterativo sobre TODOS los pares (como H2),
# luego positivos. Split RANDOM 80/10/10 por interaccion (protocolo CPGRec).
t0 = time.time(); parts = []
for chunk in pd.read_csv(recs_csv, usecols=['app_id','user_id','hours','is_recommended','date'],
                         chunksize=2_000_000,
                         dtype={'app_id':'int32','user_id':'int32','hours':'float32','is_recommended':'bool'}):
    parts.append(chunk)
recs_full = pd.concat(parts, ignore_index=True); del parts
recs_full = recs_full.drop_duplicates(['user_id','app_id'], keep='last', ignore_index=True)
print(f'Pares unicos (user,juego): {len(recs_full):,} en {time.time()-t0:.0f}s')

def iterative_k_core(df, min_user, min_game):
    it = 0
    while True:
        n0 = len(df)
        uc = df['user_id'].value_counts(); gc = df['app_id'].value_counts()
        df = df[df['user_id'].isin(uc.index[uc>=min_user]) & df['app_id'].isin(gc.index[gc>=min_game])]
        it += 1; print(f'  iter {it}: {n0:,} -> {len(df):,}')
        if len(df)==n0: return df
recs_filtered = iterative_k_core(recs_full, MIN_USER_REVIEWS, MIN_GAME_REVIEWS); del recs_full
print(f'Catalogo activo (k-core {MIN_USER_REVIEWS}/{MIN_GAME_REVIEWS}): '
      f'{recs_filtered["app_id"].nunique():,} juegos | {len(recs_filtered):,} interac')

# Positivos = is_recommended=True; playtime = horas (señal implicita)
pos = recs_filtered[recs_filtered['is_recommended']][['user_id','app_id','hours']].copy()
pos.columns = ['user_id','app_id','playtime']
pos['user_id'] = pos['user_id'].astype('int64'); pos['app_id'] = pos['app_id'].astype('int64')

# Subsample de usuarios (TIER): T1 acota; T2=None = todos (~1.9M)
if SUBSAMPLE_USERS is not None:
    _rng = np.random.default_rng(SEED)
    _u = np.sort(pos['user_id'].unique())
    keep = set(_rng.choice(_u, size=min(SUBSAMPLE_USERS, len(_u)), replace=False).tolist())
    pos = pos[pos['user_id'].isin(keep)].copy()

# Re-split RANDOM 80/10/10 por interaccion (protocolo del paper)
inter = pos.reset_index(drop=True)
_rng = np.random.default_rng(SEED)
n = len(inter); perm = _rng.permutation(n)
inter = inter.iloc[perm].reset_index(drop=True)
n_tr, n_va = int(0.8*n), int(0.1*n)
split = np.empty(n, dtype='int8'); split[:n_tr]=0; split[n_tr:n_tr+n_va]=1; split[n_tr+n_va:]=2
inter['split'] = split
print(f'inter total={len(inter):,} | usuarios={inter["user_id"].nunique():,} | juegos={inter["app_id"].nunique():,}')


Pares unicos (user,juego): 41,154,773 en 43s
  iter 1: 41,154,773 -> 22,339,679
  iter 2: 22,339,679 -> 22,294,605
  iter 3: 22,294,605 -> 22,287,486
  iter 4: 22,287,486 -> 22,287,105
  iter 5: 22,287,105 -> 22,287,029
  iter 6: 22,287,029 -> 22,286,991
  iter 7: 22,286,991 -> 22,286,979
  iter 8: 22,286,979 -> 22,286,960
  iter 9: 22,286,960 -> 22,286,960
Catalogo activo (k-core 5/20): 22,676 juegos | 22,286,960 interac
inter total=4,976,211 | usuarios=500,000 | juegos=22,590


In [ ]:
# ====== Categorias/contenido de Kozyriev (FronkonGames + games.csv + metadata) -> cat + maps ======
# Construye 'cat' con el MISMO esquema que game_categories de SCGRec, para reusar verbatim el
# escritor steam_data / SBERT / analisis. metascore <- positive_ratio (rating bien poblado de
# Kozyriev -> PER con varianza real). Guarda 'description' para el texto del SBERT.
CATALOG = sorted(int(a) for a in inter['app_id'].unique())
catalog_set = set(CATALOG); n_catalog = len(CATALOG)

def _pick_col(df, *al):
    low = {c.lower(): c for c in df.columns}
    for a in al:
        if a.lower() in low: return low[a.lower()]
    return None
def _to_int64(s): return pd.to_numeric(s, errors='coerce').astype('Int64')

# FronkonGames: genres/developers/publishers
with open(fg_csv, encoding='utf-8', errors='replace', newline='') as f:
    header = next(csv.reader(f))
fixed, repaired = [], False
for h in header:
    if h.strip().lower() in ('discountdlc count','discount dlc count'):
        fixed += ['Discount','DLC count']; repaired = True
    else: fixed.append(h)
if repaired:
    fgdf = read_csv_robust(fg_csv, engine='python', on_bad_lines='skip', dtype=str, header=0, names=fixed)
else:
    fgdf = read_csv_robust(fg_csv, engine='python', on_bad_lines='skip', dtype=str, index_col=False)
fg = pd.DataFrame()
fg['app_id'] = _to_int64(fgdf[_pick_col(fgdf,'AppID','app_id','appid','steam_appid')])
fg['genres'] = fgdf[_pick_col(fgdf,'Genres','genres')]
fg['developers'] = fgdf[_pick_col(fgdf,'Developers','developers')]
fg['publishers'] = fgdf[_pick_col(fgdf,'Publishers','publishers')]
fg = fg.dropna(subset=['app_id']); fg['app_id'] = fg['app_id'].astype('int64')
fg = fg.drop_duplicates('app_id').set_index('app_id')

# games.csv: title, price, positive_ratio, date_release
gfull = read_csv_robust(games_csv)
gfull['app_id'] = pd.to_numeric(gfull['app_id'], errors='coerce')
c_title=_pick_col(gfull,'title','name'); c_price=_pick_col(gfull,'price_final','price')
c_rat=_pick_col(gfull,'positive_ratio','rating'); c_date=_pick_col(gfull,'date_release','release_date','date')
gfull = gfull.dropna(subset=['app_id']); gfull['app_id'] = gfull['app_id'].astype('int64')
gfull = gfull.drop_duplicates('app_id').set_index('app_id')

# metadata json: description + tags (fallback de genero)
desc_map, tags_map = {}, {}
with open(meta_json, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line: continue
        try: rec = json.loads(line)
        except json.JSONDecodeError: continue
        aid = rec.get('app_id')
        if aid is None: continue
        aid = int(aid); desc_map[aid] = (rec.get('description') or '').strip(); tags_map[aid] = rec.get('tags') or []

def _split_multi(x):
    if x is None or (isinstance(x, float) and pd.isna(x)): return []
    items = [str(t) for t in x] if isinstance(x, (list, tuple, np.ndarray)) else str(x).split(',')
    return [t.strip().replace(',', ' ') for t in items if t and str(t).strip().lower() != 'nan']
def _fg(aid, col):
    try: return fg.at[aid, col]
    except Exception: return None
def _gv(aid, col):
    try: return gfull.at[aid, col] if col else None
    except Exception: return None

rows = []
for a in CATALOG:
    g = _split_multi(_fg(a, 'genres'))
    if not g: g = _split_multi(tags_map.get(a))   # fallback a koz_tags
    d = _split_multi(_fg(a, 'developers')); p = _split_multi(_fg(a, 'publishers'))
    rat = pd.to_numeric(_gv(a, c_rat), errors='coerce') if c_rat else np.nan
    pr  = pd.to_numeric(_gv(a, c_price), errors='coerce') if c_price else np.nan
    dt  = _gv(a, c_date) if c_date else None
    nm  = _gv(a, c_title) if c_title else None
    rows.append({'app_id': a,
                 'name': (str(nm) if isinstance(nm, str) and nm.strip() else f'app_{a}'),
                 'price': (float(pr) if pd.notna(pr) else np.nan),
                 'release_date': dt,
                 'metascore': (float(rat) if pd.notna(rat) else np.nan),
                 'genres': g, 'developers': d, 'publishers': p,
                 'description': desc_map.get(a, '')})
cat = pd.DataFrame(rows)
_medp = float(pd.to_numeric(cat['price'], errors='coerce').median())
_medr = float(pd.to_numeric(cat['metascore'], errors='coerce').median())
cat['price'] = cat['price'].fillna(_medp); cat['metascore'] = cat['metascore'].fillna(_medr)

# maps app_id -> list (mismo formato que H3)
genre_map = {int(a): list(g) for a, g in zip(cat['app_id'], cat['genres'])}
dev_map   = {int(a): list(g) for a, g in zip(cat['app_id'], cat['developers'])}
pub_map   = {int(a): list(g) for a, g in zip(cat['app_id'], cat['publishers'])}
total_map = {a: [('g', x) for x in genre_map.get(a, [])] + [('d', x) for x in dev_map.get(a, [])]
                + [('p', x) for x in pub_map.get(a, [])] for a in CATALOG}
CAT_MAPS = {'gene': genre_map, 'dev': dev_map, 'pub': pub_map, 'total': total_map}

# train/test + por-usuario (igual que H3)
train, test = inter[inter['split']==0], inter[inter['split']==2]
train_items_per_user = train.groupby('user_id')['app_id'].apply(set).to_dict()
test_items_per_user  = {int(u): set(int(x) for x in g) for u, g in test.groupby('user_id')['app_id']}
eval_users = [u for u in test_items_per_user if u in train_items_per_user]
print(f'cat={len(cat):,} juegos | con genero={sum(1 for a in CATALOG if genre_map.get(a)):,} | '
      f'cobertura rating(metascore<-positive_ratio)={cat["metascore"].notna().mean()*100:.0f}%')
print(f'train={len(train):,} test={len(test):,} | eval usuarios={len(eval_users):,} | '
      f'positivos/usuario (medio)={np.mean([len(test_items_per_user[u]) for u in eval_users]):.2f}')


cat=22,590 juegos | con genero=22,583 | cobertura rating(metascore<-positive_ratio)=100%
train=3,980,968 test=497,622 | eval usuarios=272,651 | positivos/usuario (medio)=1.82


In [ ]:
# Nombres de juegos (para los ejemplos del póster): app_id -> name desde game_categories.
_cx = cat.set_index('app_id')
def _gname(a):
    if a in _cx.index:
        v = _cx.loc[a].get('name')
        if isinstance(v, str) and v.strip(): return v
    return f'app_{a}'
name_map = {a: _gname(a) for a in CATALOG}
print('name_map listo:', len(name_map), 'juegos. Ej:', list(name_map.items())[:3])


name_map listo: 22590 juegos. Ej: [(10, 'Counter-Strike'), (20, 'Team Fortress Classic'), (30, 'Day of Defeat')]


In [ ]:
# Métricas del protocolo de reproducción (accuracy multi-relevante + diversidad por categoría).
def _dcg(hits):
    return sum((1.0 / math.log2(i + 2)) for i, h in enumerate(hits) if h)

def _cat_cov_ent(recs, cat_map, k):
    covs, ents = [], []
    for rec in recs.values():
        cnt = {}
        for it in rec[:k]:
            for c in cat_map.get(it, ()):
                cnt[c] = cnt.get(c, 0) + 1
        if not cnt:
            covs.append(0); ents.append(0.0); continue
        covs.append(len(cnt)); tot = sum(cnt.values())
        ents.append(-sum((v / tot) * math.log2(v / tot) for v in cnt.values()))
    return (float(np.mean(covs)) if covs else 0.0, float(np.mean(ents)) if ents else 0.0)

def paper_metrics(recs, test_items, ks=KS, cat_maps=None):
    acc = {f'{m}@{k}': [] for k in ks for m in ('Recall', 'NDCG', 'Hit', 'Precision')}
    n = 0
    for u, rec in recs.items():
        rel = test_items.get(u)
        if not rel:
            continue
        n += 1
        for k in ks:
            hits = [(1 if it in rel else 0) for it in rec[:k]]
            nhit = sum(hits)
            acc[f'Recall@{k}'].append(nhit / len(rel))
            acc[f'Precision@{k}'].append(nhit / k)
            acc[f'Hit@{k}'].append(1.0 if nhit > 0 else 0.0)
            idcg = sum(1.0 / math.log2(i + 2) for i in range(min(len(rel), k)))
            acc[f'NDCG@{k}'].append(_dcg(hits) / idcg if idcg > 0 else 0.0)
    out = {key: (float(np.mean(v)) if v else 0.0) for key, v in acc.items()}
    if cat_maps:
        for k in ks:
            for name, cmap in cat_maps.items():
                cov, ent = _cat_cov_ent(recs, cmap, k)
                out[f'Cov_{name}@{k}'] = cov; out[f'Ent_{name}@{k}'] = ent
    out['n_users'] = n
    return out

In [ ]:
pop = train.groupby('app_id').size().to_dict()
popular_list = [it for it, _ in sorted(pop.items(), key=lambda kv: (-kv[1], kv[0])) if it in catalog_set]

TOPN = max(KS)
recs_mp = {}
for u in eval_users:
    seen = train_items_per_user.get(u, set())
    recs_mp[u] = [i for i in popular_list if i not in seen][:TOPN]

m_mp = paper_metrics(recs_mp, test_items_per_user, cat_maps=CAT_MAPS)
print('Most Popular (split 80/10/10 Kozyriev):')
for k in KS:
    print(f'  @{k}: Recall={m_mp[f"Recall@{k}"]:.4f} NDCG={m_mp[f"NDCG@{k}"]:.4f} '
          f'Hit={m_mp[f"Hit@{k}"]:.4f} Prec={m_mp[f"Precision@{k}"]:.4f} '
          f'Cov(total)={m_mp[f"Cov_total@{k}"]:.2f} Ent(gene)={m_mp[f"Ent_gene@{k}"]:.3f}')

Most Popular (split 80/10/10 Kozyriev):
  @5: Recall=0.0288 NDCG=0.0207 Hit=0.0466 Prec=0.0095 Cov(total)=13.23 Ent(gene)=1.739
  @10: Recall=0.0499 NDCG=0.0282 Hit=0.0797 Prec=0.0083 Cov(total)=29.25 Ent(gene)=3.161


In [ ]:
# ====== ALS (implicit) — techo colaborativo, confianza = 1 + alpha*playtime ======
import scipy.sparse as _sp
from implicit.als import AlternatingLeastSquares
als_users = sorted(int(u) for u in train['user_id'].unique())
_u2i = {u:i for i,u in enumerate(als_users)}
_a2i = {a:i for i,a in enumerate(CATALOG)}
_tr = train[train['app_id'].isin(_a2i)]
_rows = _tr['user_id'].map(_u2i).to_numpy()
_cols = _tr['app_id'].map(_a2i).to_numpy()
_conf = (1.0 + ALS_ALPHA * np.log1p(_tr['playtime'].fillna(0).clip(lower=0).to_numpy())).astype('float32')
_ui = _sp.csr_matrix((_conf, (_rows, _cols)), shape=(len(als_users), len(CATALOG)))
als = AlternatingLeastSquares(factors=ALS_FACTORS, regularization=ALS_REG, iterations=ALS_ITERS, random_state=SEED, use_gpu=False)
als.fit(_ui)
als_uf = np.asarray(als.user_factors); als_if = np.asarray(als.item_factors)
print('ALS listo:', als_uf.shape, als_if.shape)


  0%|          | 0/15 [00:00<?, ?it/s]

ALS listo: (498829, 64) (22590, 64)


In [ ]:
# ====== eval_set (fijo) + helpers + recs de modelos SIN seed (MostPop, ALS) ======
if N_EVAL_ANALYSIS and len(eval_users) > N_EVAL_ANALYSIS:
    _rng = np.random.default_rng(SEED)
    eval_set = sorted(_rng.choice(np.array(eval_users), size=N_EVAL_ANALYSIS, replace=False).tolist())
else:
    eval_set = list(eval_users)
print(f'eval_set: {len(eval_set):,} usuarios | SEEDS={SEEDS}')

idx2app_als = {i:a for a,i in _a2i.items()}
def _topn(scores, idx2app, seen, topn):
    rec=[]
    for j in np.argsort(-scores):
        a=idx2app[j]
        if a not in seen:
            rec.append(a)
            if len(rec)>=topn: break
    return rec
def recs_emb(e_u, e_i, umap, idx2app, users, topn):
    out={}; nfb=0
    for u in users:
        seen=train_items_per_user.get(u,set()); key=str(u)
        if key not in umap:
            out[u]=[i for i in popular_list if i not in seen][:topn]; nfb+=1; continue
        out[u]=_topn(e_i @ e_u[umap[key]], idx2app, seen, topn)
    return out, nfb

recs_fixed = {}
recs_fixed['Most Popular'] = {u:[i for i in popular_list if i not in train_items_per_user.get(u,set())][:TOPN] for u in eval_set}
als_umap = {str(u):_u2i[u] for u in als_users}
recs_fixed['ALS'], _ = recs_emb(als_uf, als_if, als_umap, idx2app_als, eval_set, TOPN)
metrics_fixed = {m: paper_metrics(recs_fixed[m], test_items_per_user, cat_maps=CAT_MAPS) for m in recs_fixed}
print('recs fijas (no dependen del seed):', list(recs_fixed))


eval_set: 200,000 usuarios | SEEDS=[42]
recs fijas (no dependen del seed): ['Most Popular', 'ALS']


## Helpers beyond-accuracy (long-tail)

In [ ]:
# ====== Helpers beyond-accuracy (long-tail por actividad) — portados del corregido ======
def _bucket(n): return '2-5' if n <= 5 else '6-20' if n <= 20 else '21-50' if n <= 50 else '51+'
def _u_nr(rec, rel, k):
    hits = [1 if it in rel else 0 for it in rec[:k]]; nh = sum(hits)
    dcg = sum(1 / math.log2(i + 2) for i, h in enumerate(hits) if h)
    idcg = sum(1 / math.log2(i + 2) for i in range(min(len(rel), k)))
    return (dcg / idcg if idcg > 0 else 0.0, nh / len(rel) if rel else 0.0)
buckets = ['2-5', '6-20', '21-50', '51+']
_act = {u: _bucket(len(train_items_per_user.get(u, set()))) for u in eval_set}
def _longtail(recs_u):
    out = {}
    for b in buckets:
        us = [u for u in eval_set if _act[u] == b and test_items_per_user.get(u)]
        if not us: continue
        out[b] = {'n': len(us),
                  'NDCG@10':  float(np.mean([_u_nr(recs_u[u], test_items_per_user[u], 10)[0] for u in us])),
                  'Recall@10':float(np.mean([_u_nr(recs_u[u], test_items_per_user[u], 10)[1] for u in us]))}
    return out
def _longtail_deep(recs_u, users):
    out = {}
    for b in buckets:
        us = [u for u in users if _act.get(u) == b and test_items_per_user.get(u) and u in recs_u]
        if not us: continue
        out[b] = {'n': len(us),
                  'NDCG@10':  float(np.mean([_u_nr(recs_u[u], test_items_per_user[u], 10)[0] for u in us])),
                  'Recall@10':float(np.mean([_u_nr(recs_u[u], test_items_per_user[u], 10)[1] for u in us]))}
    return out
_clean = lambda d: {k: (float(v) if isinstance(v, (int, float, np.floating)) else v) for k, v in d.items()}
print('helpers beyond-accuracy OK | usuarios por bucket:',
      {b: sum(1 for u in eval_set if _act[u] == b) for b in buckets})


helpers beyond-accuracy OK | usuarios por bucket: {'2-5': 88748, '6-20': 93068, '21-50': 15310, '51+': 2874}


## E. Modelos feature-based (FM / DeepFM / DeepNN) — full-ranking comparable

In [ ]:
# ====== E. FM/DeepFM/DeepNN -- features + helpers (full-ranking, 5-core, SIN fuga de playtime) ======
# Mismos datos/eval_set/metricas que ALS/CPGRec del corregido -> filas directas a la tabla.
# deepctr-torch es pointwise binario: positivos implicitos (interac. de train) + neg-sampling.
import torch, time as _time
from deepctr_torch.inputs import SparseFeat, DenseFeat, VarLenSparseFeat, get_feature_names
from deepctr_torch.models import DeepFM, WDL

DEVICE_DEEP = 'cuda' if torch.cuda.is_available() else 'cpu'
EMB_DEEP    = 32
DEEP_NEG    = 4
DEEP_BATCH  = 16384
DEEP_EPOCHS = 10        if TIER == 'T2' else 1
DEEP_MAXPOS = 2_000_000 if TIER == 'T2' else 200_000      # tope de positivos de TRAIN (eval completo)
DEEP_UBATCH = 512                                          # Kozyriev ~22K items -> lote de usuarios chico

# indexacion contigua REUSANDO la de ALS (raw user -> 0..N-1 via _u2i ; app -> 0..M-1 via _a2i)
item2idx  = dict(_a2i); idx2app_d = dict(idx2app_als); N_ITEMS_D = len(item2idx)
N_USERS_D = len(als_users)

# dense (NINGUNA usa playtime): count (usuario), reccount (#is_recommended por juego), metascore (positive_ratio)
user_count = np.zeros(N_USERS_D, dtype='float32')
for _u, _c in train.groupby('user_id')['app_id'].size().items():
    if _u in _u2i: user_count[_u2i[_u]] = float(_c)
_rcd = recs_filtered.groupby('app_id')['is_recommended'].sum()
item_reccount = np.array([float(_rcd.get(a, 0.0)) for a in CATALOG], dtype='float32')
_meta = dict(zip(cat['app_id'].astype(int), cat['metascore'].astype(float)))   # = positive_ratio
item_meta = np.array([float(_meta.get(a, 0.0)) for a in CATALOG], dtype='float32')

def _z(x, logx=True):
    x = np.log1p(np.clip(x, 0, None)) if logx else x.astype('float32')
    return ((x - x.mean()) / (x.std() + 1e-6)).astype('float32')
user_count_z    = _z(user_count)
item_reccount_z = _z(item_reccount)
item_meta_z     = _z(item_meta, logx=False)

# generos -> secuencia padded por item
_allg = sorted({g for a in CATALOG for g in genre_map.get(a, [])})
genre2id = {g: k + 1 for k, g in enumerate(_allg)}            # 0 = padding
VOCAB_G, MAXLEN_G = len(_allg) + 1, 3
def _padg(a):
    ids = [genre2id[g] for g in genre_map.get(a, []) if g in genre2id][:MAXLEN_G]
    return ids + [0] * (MAXLEN_G - len(ids))
item_gseq = np.stack([_padg(a) for a in CATALOG]).astype('int64')
item_glen = np.array([max(1, min(len(genre_map.get(a, [])), MAXLEN_G)) for a in CATALOG], dtype='int64')

DENSE_COLS_D = ['count', 'reccount', 'metascore']
def _featcols_d():
    sparse = [SparseFeat('user_idx', N_USERS_D, embedding_dim=EMB_DEEP),
              SparseFeat('item_idx', N_ITEMS_D, embedding_dim=EMB_DEEP)]
    varlen = [VarLenSparseFeat(SparseFeat('genres_seq', VOCAB_G, embedding_dim=EMB_DEEP),
                               maxlen=MAXLEN_G, combiner='mean', length_name='genres_len')]
    dense  = [DenseFeat(c, 1) for c in DENSE_COLS_D]
    cols = sparse + dense + varlen
    return cols, cols

def _build_train_d(seed, max_pos=None):
    if max_pos is None: max_pos = DEEP_MAXPOS
    rng = np.random.default_rng(seed)
    tr = train[train['app_id'].isin(item2idx)]
    pu = tr['user_id'].map(_u2i).to_numpy().astype('int64')
    pi = tr['app_id'].map(item2idx).to_numpy().astype('int64')
    if max_pos is not None and len(pu) > max_pos:
        sel = rng.choice(len(pu), size=max_pos, replace=False); pu, pi = pu[sel], pi[sel]
    P  = len(pu)
    nu = np.repeat(pu, DEEP_NEG); ni = rng.integers(0, N_ITEMS_D, size=P * DEEP_NEG).astype('int64')
    u  = np.concatenate([pu, nu]); i = np.concatenate([pi, ni])
    y  = np.concatenate([np.ones(P, 'float32'), np.zeros(P * DEEP_NEG, 'float32')])
    pm = rng.permutation(len(u)); u, i, y = u[pm], i[pm], y[pm]
    X  = {'user_idx': u, 'item_idx': i, 'count': user_count_z[u],
          'reccount': item_reccount_z[i], 'metascore': item_meta_z[i],
          'genres_seq': item_gseq[i], 'genres_len': item_glen[i]}
    return X, y

def _build_model_d(kind, lin, dnn):
    if kind == 'FM':     return DeepFM(lin, dnn, dnn_hidden_units=(),         task='binary', device=DEVICE_DEEP)
    if kind == 'DeepFM': return DeepFM(lin, dnn, dnn_hidden_units=(8, 8),  dnn_dropout=0.2, task='binary', device=DEVICE_DEEP)
    if kind == 'DeepNN': return WDL([],  dnn, dnn_hidden_units=(32, 32), dnn_dropout=0.2, task='binary', device=DEVICE_DEEP)
    raise ValueError(kind)

def _deep_full_rank(model, users, topn, ubatch=None, log_every=20):
    if ubatch is None: ubatch = DEEP_UBATCH
    out = {}; all_i = np.arange(N_ITEMS_D, dtype='int64'); nb = (len(users) + ubatch - 1) // ubatch
    for bi_, s in enumerate(range(0, len(users), ubatch)):
        ub_raw = list(users[s:s + ubatch]); ub = np.array([_u2i[u] for u in ub_raw], dtype='int64'); B = len(ub)
        u_rep = np.repeat(ub, N_ITEMS_D); i_rep = np.tile(all_i, B)
        X = {'user_idx': u_rep, 'item_idx': i_rep, 'count': user_count_z[u_rep],
             'reccount': item_reccount_z[i_rep], 'metascore': item_meta_z[i_rep],
             'genres_seq': item_gseq[i_rep], 'genres_len': item_glen[i_rep]}
        sc = model.predict(X, batch_size=131072).reshape(B, N_ITEMS_D)
        for bi in range(B):
            u = ub_raw[bi]; seen = train_items_per_user.get(u, set()); rec = []
            for j in np.argsort(-sc[bi]):
                a = idx2app_d[int(j)]
                if a not in seen:
                    rec.append(a)
                    if len(rec) >= topn: break
            out[u] = rec
        if log_every and (bi_ % log_every == 0):
            print(f'       full-rank lote {bi_+1}/{nb}', flush=True)
    return out

print(f'deep listo: N_USERS={N_USERS_D} N_ITEMS={N_ITEMS_D} generos={len(_allg)} | TIER={TIER} '
      f'epochs={DEEP_EPOCHS} neg={DEEP_NEG} maxpos={DEEP_MAXPOS} batch={DEEP_BATCH} device={DEVICE_DEEP}')


deep listo: N_USERS=498829 N_ITEMS=22590 generos=404 | TIER=T2 epochs=10 neg=4 maxpos=2000000 batch=16384 device=cuda


In [ ]:
# ====== E. Loop multi-seed FM/DeepFM/DeepNN (entrena + full-ranking + paper_metrics + long-tail) ======
import json as _json, time as _time
DEEP_VARIANTS = ['FM', 'DeepFM', 'DeepNN']
DEEP_SEEDS = SEEDS if TIER == 'T2' else SEEDS[:1]
DEEP_EVAL  = list(eval_set) if TIER == 'T2' else list(eval_set)[:2000]
_lin_d, _dnn_d = _featcols_d()
deep_per_seed = []; deep_seed0_recs = {}
os.makedirs('deep_kozyriev_h3', exist_ok=True)
_PARTIAL_D = 'deep_kozyriev_h3/deep_per_seed_partial.json'
print(f'DEEP run: TIER={TIER} seeds={DEEP_SEEDS} epochs={DEEP_EPOCHS} eval_users={len(DEEP_EVAL):,}', flush=True)
for _si, _s in enumerate(DEEP_SEEDS):
    print(f'\n========== DEEP SEED {_s} ({_si+1}/{len(DEEP_SEEDS)}) ==========', flush=True)
    torch.manual_seed(_s); np.random.seed(_s)
    _Xtr, _ytr = _build_train_d(_s)
    print(f'  train: {len(_ytr):,} muestras ({int(_ytr.sum()):,} pos)', flush=True)
    _rec = {'seed': _s, 'longtail': {}}
    for _k in DEEP_VARIANTS:
        torch.manual_seed(_s)
        _m = _build_model_d(_k, _lin_d, _dnn_d)
        _m.compile('adam', 'binary_crossentropy', metrics=[])
        print(f'  [{_k}] entrenando {DEEP_EPOCHS} ep (batch {DEEP_BATCH})...', flush=True)
        for _ep in range(DEEP_EPOCHS):
            _t0 = _time.time()
            _h = _m.fit(_Xtr, _ytr, batch_size=DEEP_BATCH, epochs=1, verbose=0, shuffle=True)
            _ls = _h.history.get('loss', [float('nan')])[-1]
            print(f'     {_k} ep {_ep+1}/{DEEP_EPOCHS}  loss={_ls:.4f}  ({_time.time()-_t0:.0f}s)', flush=True)
        print(f'  [{_k}] full-ranking sobre {len(DEEP_EVAL):,} usuarios...', flush=True)
        _r = _deep_full_rank(_m, DEEP_EVAL, TOPN)
        _rec[_k] = _clean(paper_metrics(_r, test_items_per_user, cat_maps=CAT_MAPS))
        _rec['longtail'][_k] = _longtail_deep(_r, DEEP_EVAL)
        if _si == 0: deep_seed0_recs[_k] = _r
        print(f'   -> {_k:7s}: R@5={_rec[_k]["Recall@5"]:.4f} NDCG@5={_rec[_k]["NDCG@5"]:.4f} '
              f'NDCG@10={_rec[_k]["NDCG@10"]:.4f}', flush=True)
        del _m
        if DEVICE_DEEP == 'cuda': torch.cuda.empty_cache()
    deep_per_seed.append(_rec); _json.dump(deep_per_seed, open(_PARTIAL_D, 'w'), indent=2)
    print(f'[deep seed {_s}] FM R@5={_rec["FM"]["Recall@5"]:.4f} | '
          f'DeepFM R@5={_rec["DeepFM"]["Recall@5"]:.4f} | DeepNN R@5={_rec["DeepNN"]["Recall@5"]:.4f}', flush=True)
print(f'\nDeep multi-seed completo: {len(deep_per_seed)}/{len(DEEP_SEEDS)} seeds.')


DEEP run: TIER=T2 seeds=[42] epochs=10 eval_users=200,000

========== DEEP SEED 42 (1/1) ==========
  train: 10,000,000 muestras (2,000,000 pos)
  [FM] entrenando 10 ep (batch 16384)...
cuda
Train on 10000000 samples, validate on 0 samples, 611 steps per epoch
     FM ep 1/10  loss=0.3381  (110s)
cuda
Train on 10000000 samples, validate on 0 samples, 611 steps per epoch
     FM ep 2/10  loss=0.2357  (109s)
cuda
Train on 10000000 samples, validate on 0 samples, 611 steps per epoch
     FM ep 3/10  loss=0.2234  (109s)
cuda
Train on 10000000 samples, validate on 0 samples, 611 steps per epoch
     FM ep 4/10  loss=0.2103  (109s)
cuda
Train on 10000000 samples, validate on 0 samples, 611 steps per epoch
     FM ep 5/10  loss=0.1965  (109s)
cuda
Train on 10000000 samples, validate on 0 samples, 611 steps per epoch
     FM ep 6/10  loss=0.1838  (109s)
cuda
Train on 10000000 samples, validate on 0 samples, 611 steps per epoch
     FM ep 7/10  loss=0.1728  (109s)
cuda
Train on 10000000 samples

## A. Accuracy (este notebook + filas CPGRec del corregido)

In [ ]:
# ====== A. Accuracy (MostPop/ALS de este notebook + FM/DeepFM/DeepNN media+-std) ======
# Comparable con las filas CPGRec de RESULTADOS_kozyriev_corregido.md (MISMO eval_set/metricas/split).
_MET = ['Recall@5', 'NDCG@5', 'Hit@5', 'Precision@5', 'Recall@10', 'NDCG@10']
def _ms(vs):
    return float(np.mean(vs)), (float(np.std(vs)) if len(vs) > 1 else 0.0)
rows = []
for name in ['Most Popular', 'ALS']:
    d = metrics_fixed[name]; rows.append([name] + [f'{d[m]:.4f}' for m in _MET])
for k in ['FM', 'DeepFM', 'DeepNN']:
    cells = []
    for m in _MET:
        mm, ss = _ms([r[k][m] for r in deep_per_seed]); cells.append(f'{mm:.4f}\u00b1{ss:.4f}')
    rows.append([k] + cells)
dfA = pd.DataFrame(rows, columns=['Modelo'] + _MET)
delA = {}                                          # V2 no tiene variantes pareadas (deltas) como el GNN
print('=== A Accuracy (media+-std sobre seeds) ===')
print(dfA.to_string(index=False))
print('\n> MostPop/ALS deben reproducir (self-check) sus filas del corregido (mismo eval_set).')
print('> Pegar las filas de CPGRec base/+PER/+PER+PRG desde RESULTADOS_kozyriev_corregido.md.')
dfA


=== A Accuracy (media+-std sobre seeds) ===
      Modelo      Recall@5        NDCG@5         Hit@5   Precision@5     Recall@10       NDCG@10
Most Popular        0.0289        0.0207        0.0466        0.0095        0.0500        0.0282
         ALS        0.0586        0.0436        0.0927        0.0194        0.0931        0.0559
          FM 0.0239±0.0000 0.0169±0.0000 0.0391±0.0000 0.0080±0.0000 0.0416±0.0000 0.0232±0.0000
      DeepFM 0.0263±0.0000 0.0186±0.0000 0.0431±0.0000 0.0088±0.0000 0.0460±0.0000 0.0255±0.0000
      DeepNN 0.0092±0.0000 0.0064±0.0000 0.0168±0.0000 0.0034±0.0000 0.0181±0.0000 0.0096±0.0000

> MostPop/ALS deben reproducir (self-check) sus filas del corregido (mismo eval_set).
> Pegar las filas de CPGRec base/+PER/+PER+PRG desde RESULTADOS_kozyriev_corregido.md.


,Modelo,Recall@5,NDCG@5,Hit@5,Precision@5,Recall@10,NDCG@10
0,Most Popular,0.0289,0.0207,0.0466,0.0095,0.0500,0.0282
1,ALS,0.0586,0.0436,0.0927,0.0194,0.0931,0.0559
2,FM,0.0239±0.0000,0.0169±0.0000,0.0391±0.0000,0.0080±0.0000,0.0416±0.0000,0.0232±0.0000
3,DeepFM,0.0263±0.0000,0.0186±0.0000,0.0431±0.0000,0.0088±0.0000,0.0460±0.0000,0.0255±0.0000
4,DeepNN,0.0092±0.0000,0.0064±0.0000,0.0168±0.0000,0.0034±0.0000,0.0181±0.0000,0.0096±0.0000


## B. Diversidad (Cov/Ent a nivel categoría)

In [ ]:
# ====== B. Diversidad (Cov/Ent categoria; deep media+-std) ======
DIV = [f'Cov_total@{k}' for k in KS] + [f'Cov_gene@{k}' for k in KS] + [f'Ent_gene@{k}' for k in KS]
rowsB = {}
for m in ('Most Popular', 'ALS'):
    rowsB[m] = {c: f'{metrics_fixed[m][c]:.4f}' for c in DIV}
for m in ['FM', 'DeepFM', 'DeepNN']:
    _bd = {c: (float(np.mean([ps[m][c] for ps in deep_per_seed])),
               float(np.std([ps[m][c] for ps in deep_per_seed]))) for c in DIV}
    rowsB[m] = {c: f'{_bd[c][0]:.4f}\u00b1{_bd[c][1]:.4f}' for c in DIV}
dfB = pd.DataFrame(rowsB).T[DIV]
print('=== B Diversidad (Cov=nro categorias distintas en top-K; Ent=entropia; media+-std) ===\n')
print(dfB.to_string())
print('\n> Filas CPGRec: pegar de RESULTADOS_kozyriev_corregido.md (mismo eval_set).')


=== B Diversidad (Cov=nro categorias distintas en top-K; Ent=entropia; media+-std) ===

                 Cov_total@5    Cov_total@10     Cov_gene@5     Cov_gene@10     Ent_gene@5    Ent_gene@10
Most Popular         13.2255         29.2414         4.0198         10.8365         1.7374         3.1607
ALS                  18.4826         31.9133         6.7256          9.3655         2.3930         2.7240
FM            15.6125±0.0000  28.1953±0.0000  5.0001±0.0000   7.9775±0.0000  1.5372±0.0000  2.0796±0.0000
DeepFM        15.3668±0.0000  27.8995±0.0000  4.7523±0.0000   7.4106±0.0000  1.6333±0.0000  2.1354±0.0000
DeepNN        18.3161±0.0000  32.8991±0.0000  6.9803±0.0000  10.1273±0.0000  2.4515±0.0000  2.8463±0.0000

> Filas CPGRec: pegar de RESULTADOS_kozyriev_corregido.md (mismo eval_set).


## C. Long-tail (NDCG@10 por actividad del usuario)

In [ ]:
# ====== C. Long-tail: NDCG@10 por actividad del usuario (deep media+-std) ======
lt_fixed = {m: _longtail(recs_fixed[m]) for m in recs_fixed}
order = ['Most Popular', 'ALS', 'FM', 'DeepFM', 'DeepNN']
rowsC = []; nrow = {}
for b in buckets:
    row = {'actividad': b}
    for m in ('Most Popular', 'ALS'):
        row[m] = (f'{lt_fixed[m][b]["NDCG@10"]:.4f}' if b in lt_fixed[m] else '')
    for m in ['FM', 'DeepFM', 'DeepNN']:
        vals = [ps['longtail'][m][b]['NDCG@10'] for ps in deep_per_seed if b in ps['longtail'][m]]
        row[m] = (f'{np.mean(vals):.4f}\u00b1{np.std(vals):.4f}' if vals else '')
    rowsC.append(row); nrow[b] = lt_fixed['Most Popular'].get(b, {}).get('n', '?')
dfC = pd.DataFrame(rowsC)[['actividad'] + order]
print('=== C Long-tail: NDCG@10 por actividad (media+-std) ===')
print(dfC.to_string(index=False))
print('\nn usuarios por bucket:')
for b in buckets: print(f'  {b:6s}: {nrow[b]}')


=== C Long-tail: NDCG@10 por actividad (media+-std) ===
actividad Most Popular    ALS            FM        DeepFM        DeepNN
      2-5       0.0292 0.0534 0.0234±0.0000 0.0261±0.0000 0.0088±0.0000
     6-20       0.0277 0.0585 0.0231±0.0000 0.0250±0.0000 0.0093±0.0000
    21-50       0.0258 0.0551 0.0226±0.0000 0.0248±0.0000 0.0140±0.0000
      51+       0.0244 0.0534 0.0234±0.0000 0.0270±0.0000 0.0206±0.0000

n usuarios por bucket:
  2-5   : 88748
  6-20  : 93068
  21-50 : 15310
  51+   : 2874


## D. Ejemplos de recomendaciones (con nombres, para el póster)

In [ ]:
# ====== D. Ejemplos de recomendaciones (seed representativo = primer seed deep) ======
recs = {'Most Popular': recs_fixed['Most Popular'], 'ALS': recs_fixed['ALS'],
        'FM': deep_seed0_recs['FM'], 'DeepFM': deep_seed0_recs['DeepFM'], 'DeepNN': deep_seed0_recs['DeepNN']}
def _nm(items): return [name_map.get(a, str(a)) for a in items]
_chosen = []
for b in buckets:
    for u in [x for x in eval_set if _act[x] == b and test_items_per_user.get(x)]:
        if all(u in recs[m] for m in recs) and any(any(it in test_items_per_user[u] for it in recs[m][u][:5]) for m in recs):
            _chosen.append((b, u)); break
    if len(_chosen) >= N_EXAMPLES: break
ej = [f'(ejemplos del seed {DEEP_SEEDS[0]})']
for b, u in _chosen[:N_EXAMPLES]:
    rel = test_items_per_user[u]; hist = sorted(train_items_per_user.get(u, set()))
    ej.append(f'\n### Usuario {u}  (actividad {b}, {len(hist)} juegos en historial)')
    ej.append('- Perfil (muestra): ' + ', '.join(_nm(hist[:6])))
    ej.append('- Test (a acertar): ' + ', '.join(_nm(sorted(rel))))
    for m in recs:
        nd, rc = _u_nr(recs[m][u], rel, 5)
        marks = [name_map.get(a, str(a)) + (' \u2713' if a in rel else '') for a in recs[m][u][:5]]
        ej.append(f'  - **{m}** (R@5={rc:.2f}, NDCG@5={nd:.2f}): ' + ', '.join(marks))
ej_txt = '\n'.join(ej)
print('=== D Ejemplos ===\n' + ej_txt)


=== D Ejemplos ===
(ejemplos del seed 42)

### Usuario 2107  (actividad 2-5, 5 juegos en historial)
- Perfil (muestra): Fallout: New Vegas, Borderlands 2, Life is Strange: Before the Storm, Among Us, Halo: The Master Chief Collection
- Test (a acertar): Life is Strange - Episode 1
  - **Most Popular** (R@5=0.00, NDCG@5=0.00): Team Fortress 2, The Witcher® 3: Wild Hunt, DARK SOULS™ III, Cyberpunk 2077, Left 4 Dead 2
  - **ALS** (R@5=1.00, NDCG@5=1.00): Life is Strange - Episode 1 ✓, Wallpaper Engine, Destiny 2, Crusader Kings II, Life is Strange 2
  - **FM** (R@5=0.00, NDCG@5=0.00): DARK SOULS™ III, Left 4 Dead 2, Deep Rock Galactic, The Witcher® 3: Wild Hunt, Cyberpunk 2077
  - **DeepFM** (R@5=0.00, NDCG@5=0.00): DARK SOULS™ III, Team Fortress 2, Wallpaper Engine, Left 4 Dead 2, Cyberpunk 2077
  - **DeepNN** (R@5=0.00, NDCG@5=0.00): Batman™: Arkham Origins, Call of Duty®: Black Ops II, Wolfenstein: The New Order, ACE COMBAT™ 7: SKIES UNKNOWN, Torchlight II

### Usuario 464  (actividad 

## Guardar + imprimir todo (respaldo sin descarga)

In [ ]:
# ====== Guardar (chicos) + IMPRIMIR TODO (respaldo si no se pueden descargar) ======
import json as _json
_out = 'deep_kozyriev_h3'; os.makedirs(_out, exist_ok=True)
dfA.to_csv(f'{_out}/accuracy.csv'); dfB.to_csv(f'{_out}/diversidad.csv'); dfC.to_csv(f'{_out}/longtail.csv', index=False)
open(f'{_out}/ejemplos.md', 'w', encoding='utf-8').write(ej_txt)
_full = {'seeds': SEEDS, 'n_eval': len(eval_set), 'tier': TIER, 'deep_per_seed': deep_per_seed,
         'fixed': {m: _clean(metrics_fixed[m]) for m in metrics_fixed}}
_json.dump(_full, open(f'{_out}/metricas_full.json', 'w'), indent=2)
print('Guardado en', _out, '->', sorted(os.listdir(_out)))
print('\n' + '#' * 72)
print('# RESPALDO EN TEXTO  (si no se descargan los archivos, todo queda impreso aqui)')
print('#' * 72)
print('\n===== accuracy.csv =====\n' + dfA.to_csv())
print('\n===== diversidad.csv =====\n' + dfB.to_csv())
print('\n===== longtail.csv =====\n' + dfC.to_csv(index=False))
print('\n===== ejemplos.md =====\n' + ej_txt)
print('\n===== metricas_full.json =====\n' + _json.dumps(_full, indent=2))
try:
    import shutil as _s2; _s2.make_archive(_out, 'zip', _out)
    from google.colab import files; files.download(_out + '.zip')
    print('\n(zip de descarga generado)')
except Exception as e:
    print('\n(descarga automatica no disponible:', repr(e), '-> todo esta impreso arriba)')


Guardado en deep_kozyriev_h3 -> ['accuracy.csv', 'deep_per_seed_partial.json', 'diversidad.csv', 'ejemplos.md', 'longtail.csv', 'metricas_full.json']

########################################################################
# RESPALDO EN TEXTO  (si no se descargan los archivos, todo queda impreso aqui)
########################################################################

===== accuracy.csv =====
,Modelo,Recall@5,NDCG@5,Hit@5,Precision@5,Recall@10,NDCG@10
0,Most Popular,0.0289,0.0207,0.0466,0.0095,0.0500,0.0282
1,ALS,0.0586,0.0436,0.0927,0.0194,0.0931,0.0559
2,FM,0.0239±0.0000,0.0169±0.0000,0.0391±0.0000,0.0080±0.0000,0.0416±0.0000,0.0232±0.0000
3,DeepFM,0.0263±0.0000,0.0186±0.0000,0.0431±0.0000,0.0088±0.0000,0.0460±0.0000,0.0255±0.0000
4,DeepNN,0.0092±0.0000,0.0064±0.0000,0.0168±0.0000,0.0034±0.0000,0.0181±0.0000,0.0096±0.0000


===== diversidad.csv =====
,Cov_total@5,Cov_total@10,Cov_gene@5,Cov_gene@10,Ent_gene@5,Ent_gene@10
Most Popular,13.2255,29.2414,4.0198,10.8365,1.7374,3.160

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


(zip de descarga generado)
